# Volume-weighted temperature-salinity diagram

A T-S diagram weighted by cell volume.

The original uses `xhistogram`; here it is `numpy.histogram2d`
with volume weights, which conserves the total volume exactly.

The axes are TEOS-10 quantities, as in the original: practical
salinity `so` becomes absolute salinity and potential temperature
`thetao` becomes conservative temperature. Plotting CMOR's
potential temperature against absolute salinity would misplace the
water masses relative to the sigma2 contours drawn over them.

`volcello` is optional: without it the volume is layer thickness
times `areacello`, which is what the original recommends for MOM5
output that has no `volcello` either.

### Bins, and why they are not fixed

The original builds its bins from the data:

```python
temp_bins = np.arange(np.floor(T.min())-0.5, np.ceil(T.max())+0.6, 0.5)
salt_bins = np.arange(np.floor(S.min())-0.1, np.ceil(S.max())+0.11, 0.1)
```

Both parts of that matter. The **range** comes from the data, so
the axes frame the water masses instead of leaving most of the
plot empty. And the **widths** — 0.5 °C, 0.1 g/kg — are coarse on
purpose.

Finer bins do not give more detail here, they give *holes*. A
T-S histogram can only populate as many bins as the data has
distinct values, and a time-averaged section has one T and one S
per cell, with the vertical levels setting the spacing: a
50-level column gives about 50 distinct salinities. Bin the
salinity axis at 0.025 g/kg and most bins fall between two
available values and stay empty — white streaks *through* the
distribution, which look like structure and are not.

### Why the time axis is kept

For the same reason, and one better one. This notebook does not
call `climate_statistics`: the original histograms every month,
not a climatology.

Averaging first would collapse the seasonal cycle, and the
surface end of a T-S curve is precisely where that cycle lives —
you lose the summer-warm, winter-cool spread and the diagram
contracts towards the mean water masses. Keeping the months also
multiplies the number of distinct values, which is what lets the
bins fill.

The histogram is divided by the number of steps so the colour bar
stays a volume rather than becoming volume x months.

---

Translated from the COSIMA recipe
[`Temperature_Salinity_Diagram.ipynb`](https://github.com/COSIMA/cosima-recipes/blob/main/03-Advanced-Recipes/Temperature_Salinity_Diagram.ipynb)
onto CMORised CMIP6 data, using ESMValCore's `Dataset` and
preprocessors in place of the ACCESS-NRI intake catalog.

This notebook is standalone: it needs only `esmvalcore`,
`iris`, `numpy`, `matplotlib`, `cartopy` and `gsw`. The
equivalent ESMValTool recipe is
`recipes/recipe_cosima_*.yml`.

In [ ]:
import numpy as np
import iris
import iris.analysis.cartography
import iris.plot as iplt
import matplotlib.pyplot as plt

from esmvalcore.dataset import Dataset
from dask import compute
import gsw
import matplotlib.colors as mcolors
from esmvalcore.preprocessor import extract_region, regrid

## Start a dask cluster

Every COSIMA recipe opens with one, and these need one for the same
reason, though it is worth being precise about *which* part it speeds up.

**What the cluster does accelerate.** Everything from `Dataset.load()`
through the ESMValCore preprocessors — reading multi-decade files,
`regrid`, `climate_statistics`, `extract_region` — is lazy and chunked.
That is the expensive part of these notebooks and it is what the workers
parallelise. Without a client, dask falls back to its threaded scheduler;
ESMValCore itself warns that the distributed scheduler is preferable.

**What it does not.** Once the preprocessed cube is reduced, the
diagnostic arithmetic realises it into a plain masked array and proceeds
eagerly in numpy. That is deliberate: contour tracing, per-column
remapping and density binning are iterative and index-based, so they do
not express well as array graphs and would be slower, not faster, spread
over workers. Realising *after* the time-mean is what keeps that
affordable — realising a 20-year daily 3-D field before reducing it would
not be.

So: size the cluster for the load-and-preprocess stage. One thread per
worker, as the COSIMA recipes use — with chunks this large, per-worker
memory binds well before thread count does.

On ARE, `Client(threads_per_worker=1)` picks up the cores you requested
for the session. Pass `n_workers=` and `memory_limit=` to be explicit.

**Running the equivalent recipe instead?** Do not start a client there.
ESMValTool manages dask itself, and it defaults to the threaded
scheduler; switch it on in your ESMValTool configuration:

```yaml
dask:
  use: local_distributed
```

In [ ]:
import logging
import os

from dask.distributed import Client

# distributed logs every worker registration and teardown at INFO,
# which is hundreds of lines per notebook and buries everything the
# diagnostic prints. Warnings and errors still come through.
for _name in ("distributed", "distributed.scheduler", "distributed.nanny",
              "distributed.worker", "distributed.core",
              "distributed.batched", "distributed.utils_perf"):
    logging.getLogger(_name).setLevel(logging.WARNING)

# Client.current() reuses a cluster if this cell is re-run, instead of
# quietly starting a second one alongside the first.
try:
    worker_count = max(1, int(os.environ.get("PBS_NCPUS", "1")))
except ValueError:
    worker_count = 1
    print("PBS_NCPUS is not an integer; using one worker")
try:
    client = Client.current()
except ValueError:
    try:
        # Separate worker processes: real memory isolation, and what
        # you want on ARE or any batch node.
        client = Client(n_workers=worker_count, threads_per_worker=1)
    except (RuntimeError, OSError) as error:
        # Some containers and locked-down environments cannot spawn
        # worker subprocesses. In-process workers still give a
        # distributed scheduler and a dashboard, just no isolation.
        print(f'process workers unavailable ({error}); '
              f'using in-process workers instead')
        client = Client(n_workers=worker_count, threads_per_worker=1,
                        processes=False)

active_workers = len(client.scheduler_info()['workers'])
print('dashboard:', client.dashboard_link)
print(f'workers: {active_workers} active (requested {worker_count})')
client

## Load the data

In [ ]:
common = dict(project='CMIP6', mip='Omon', exp='historical',
              ensemble='r1i1p1f1', grid='gn',
              timerange='1990/2009')

thetao = Dataset(short_name='thetao', dataset='ACCESS-CM2',
                 **common)
thetao.add_supplementary(short_name='volcello', mip='Ofx')
so = thetao.copy(short_name='so')

def prepare(dataset):
    """Regrid and subset. NO time mean -- see below."""
    cube = regrid(dataset.load(), target_grid='1x1',
                  scheme='linear')
    return extract_region(cube, start_longitude=330.,
                          end_longitude=340.,
                          start_latitude=-90., end_latitude=-37.)

thetao_cube = prepare(thetao)
so_cube = prepare(so)
# Fetch both fields through one graph.  Computing them
# independently duplicates the regrid and file-read work.
thetao_data, so_data = compute(thetao_cube.core_data(),
                                so_cube.core_data())
thetao_cube.data = thetao_data
so_cube.data = so_data
print('time steps kept:', thetao_cube.coord('time').shape[0])

## Helpers

In [ ]:
# The helpers below are lifted verbatim from
# cosima_cmip/cosima_common.py by notebooks/build_notebooks.py, so
# this notebook runs exactly the same code as the ESMValTool
# diagnostic script. Edit them there, then rebuild.
import contextlib
import warnings

try:  # TEOS-10, needed by the density helpers
    import gsw
except ImportError:
    gsw = None


import logging

LOGGER = logging.getLogger("cosima_cmip")


EARTH_RADIUS = 6371000.0  # m, matches iris.analysis.cartography


FILL_VALUE_THRESHOLD = 1.0e19


def masked_data(cube, dtype=float):
    """Realised, fully masked data of a cube.

    Always go through this rather than ``cube.core_data()``.  The latter
    returns a *lazy* dask array, on which ``numpy.ma`` operations are
    silently no-ops: the mask is not applied, and fill values flow
    straight into the arithmetic.  This realises the data, applies any
    mask the file carries, and additionally masks non-finite values and
    unconverted fill values, which real CMOR files do contain.
    """
    data = np.ma.masked_invalid(np.ma.asarray(cube.data, dtype=dtype))
    return np.ma.masked_where(
        np.abs(np.ma.filled(data, 0.0)) >= FILL_VALUE_THRESHOLD, data)


def _coord_points(cube, name):
    """1D points for ``name``, taking column/row 0 of a 2D aux coord."""
    coord = cube.coord(name)
    if coord.ndim == 1:
        return coord.points
    points = coord.points
    # On a curvilinear grid latitude varies down a column and longitude
    # along a row; take the slice that actually varies.
    if name == "latitude":
        return points[:, 0]
    return points[0, :]


def lat_1d(cube):
    """Representative 1D latitude, also for 2D (curvilinear) coordinates."""
    return _coord_points(cube, "latitude")


def lon_1d(cube):
    """Representative 1D longitude, also for 2D coordinates."""
    return _coord_points(cube, "longitude")


def guess_bounds(cube, coords=("latitude", "longitude")):
    """Add bounds in place where they are missing."""
    for name in coords:
        if cube.coords(name) and not cube.coord(name).has_bounds():
            try:
                cube.coord(name).guess_bounds()
            except ValueError:  # scalar or single-point coordinate
                pass
    return cube


def depth_coord_name(cube):
    """Name of the vertical coordinate, or None for a 2D field."""
    for name in ("depth", "ocean_sigma_z", "olevel", "lev"):
        if cube.coords(name):
            return name
    for coord in cube.coords(dim_coords=True):
        if coord.units.is_convertible("m") and coord.name() != "latitude":
            return coord.name()
    return None


def horizontal_slice(cube):
    """First slice of a cube over its two horizontal dimensions.

    ``cube.slices(["latitude", "longitude"])`` cannot be used here: on a
    curvilinear grid the two coordinates share both dimensions and iris
    rejects them as non-orthogonal.  Slicing by dimension index works
    for regular and curvilinear grids alike.
    """
    horizontal_dims = sorted(
        set(cube.coord_dims("latitude")) | set(cube.coord_dims("longitude")))
    if cube.ndim == len(horizontal_dims):
        return cube
    index = tuple(slice(None) if dim in horizontal_dims else 0
                  for dim in range(cube.ndim))
    return cube[index]


def spherical_polygon_area(lon_vertices, lat_vertices, radius=EARTH_RADIUS):
    """Area in m2 of spherical polygons given their corner coordinates.

    ``lon_vertices`` and ``lat_vertices`` are in degrees with the corners
    along the last axis, which is the ``(..., 4)`` layout CMOR uses for
    ``vertices_longitude`` / ``vertices_latitude``.  Any number of
    corners works as long as they are ordered around the cell.

    The area is the spherical excess, accumulated over a triangle fan
    from the first corner, using the Van Oosterom & Strackee (1983)
    form:

        tan(E/2) = |v1 . (v2 x v3)|
                   / (1 + v1.v2 + v2.v3 + v3.v1)

    with ``v`` the unit vectors of the corners.  This is *exact* for
    great-circle edges and therefore invariant under rotation of the
    sphere, which matters because a tripolar grid is exactly a rotated
    and stretched one: a formula that is only correct for cells aligned
    with latitude circles would degrade precisely where the grid stops
    being aligned.

    The obvious alternative, the spherical shoelace formula
    ``R^2/2 sum (dlon)(sin lat_i + sin lat_{i+1})``, treats each edge as
    linear in ``(lon, sin lat)`` rather than as a great circle.  It is
    exact for a latitude-longitude box and accurate to second order for
    small cells, but it is not rotation invariant and it fails badly for
    cells spanning large longitude ranges, such as those beside a
    displaced pole.

    The two disagree by about 2e-4 relative on a 2-degree cell, and
    neither is "wrong": they assume different cell edges.  A cell whose
    north and south edges follow *parallels* has the shoelace area,
    which is what ``iris.analysis.cartography.area_weights`` returns; a
    cell with great-circle edges has this one.  Both tile the sphere
    exactly.  The discrepancy is far smaller than the difference
    between either and the model's own ``areacello``, which is why
    ``areacello`` is still preferred when it is published.
    """
    lon = np.deg2rad(np.asarray(lon_vertices, dtype=float))
    lat = np.deg2rad(np.asarray(lat_vertices, dtype=float))
    unit = np.stack([np.cos(lat) * np.cos(lon),
                     np.cos(lat) * np.sin(lon),
                     np.sin(lat)], axis=-1)

    first = unit[..., 0, :]
    excess = np.zeros(unit.shape[:-2])
    for corner in range(1, unit.shape[-2] - 1):
        second = unit[..., corner, :]
        third = unit[..., corner + 1, :]
        triple = np.abs(np.einsum("...i,...i->...",
                                  first, np.cross(second, third)))
        denominator = (1.0
                       + np.einsum("...i,...i->...", first, second)
                       + np.einsum("...i,...i->...", second, third)
                       + np.einsum("...i,...i->...", third, first))
        excess += 2.0 * np.arctan2(triple, denominator)
    return excess * radius**2


def area_from_bounds(cube, radius=EARTH_RADIUS):
    """Cell area in m2 from 2D coordinate bounds, or ``None``.

    This is what makes a tripolar grid workable without ``areacello``.
    CMOR requires bounds on ``latitude`` and ``longitude``, and on a
    curvilinear grid those bounds are the four corner vertices of every
    cell, so the area follows exactly from
    :func:`spherical_polygon_area`.

    ESMValCore does not do this: ``try_adding_calculated_cell_area``
    handles regular and rotated-pole grids and raises
    ``CoordinateMultiDimError`` for everything else.  The information is
    in the file regardless.

    Returns ``None`` when the cube has no usable 2D bounds, so callers
    can fall back.
    """
    horizontal = horizontal_slice(cube)
    lat_coord = horizontal.coord("latitude")
    lon_coord = horizontal.coord("longitude")
    if lat_coord.ndim != 2 or lon_coord.ndim != 2:
        return None
    if lat_coord.bounds is None or lon_coord.bounds is None:
        return None
    if lat_coord.bounds.shape[-1] < 3:
        return None
    return np.ma.masked_invalid(
        spherical_polygon_area(lon_coord.bounds, lat_coord.bounds,
                               radius=radius))


def cell_area(cube):
    """Cell area in m2, preferring the model's own ``areacello``.

    Order of preference:

    1. an ``areacello`` cell measure attached to the cube, which is what
       ESMValCore builds from a ``supplementary_variables`` entry.  This
       is the model's own area and is always the best answer: it
       accounts for the actual grid generation, including any partial
       cells at the coast.
    2. a spherical polygon area from the 2D corner vertices, via
       :func:`area_from_bounds`.  CMOR mandates bounds on latitude and
       longitude, so this works on a tripolar grid with no
       ``areacello`` at all.  It assumes great-circle cell edges on a
       sphere, so it differs from the model's own areas by well under a
       percent for an ocean grid.
    3. spherical areas from 1D latitude/longitude bounds, for a regular
       grid.

    Only if all three fail does this raise, and then with instructions.
    """
    horizontal = horizontal_slice(cube)

    for measure in horizontal.cell_measures():
        if measure.var_name in ("areacello", "areacella", "areacell"):
            return np.ma.masked_invalid(
                np.ma.asarray(
                    horizontal.cell_measure(measure.name()).core_data(),
                    dtype=float).reshape(horizontal.shape))

    from_bounds = area_from_bounds(horizontal)
    if from_bounds is not None:
        return from_bounds

    if horizontal.coord("latitude").ndim > 1:
        raise ValueError(
            "this cube has 2D (curvilinear) latitude/longitude and no cell "
            "corner bounds, so no cell area can be obtained. Either attach "
            "areacello -- in a recipe with\n"
            "    supplementary_variables:\n"
            "      - short_name: areacello\n"
            "        mip: Ofx\n"
            "(add 'exp: piControl' if the experiment you want does not "
            "publish it), in a notebook with "
            "dataset.add_supplementary(short_name='areacello', mip='Ofx') "
            "-- or keep the coordinate bounds, which CMOR requires and "
            "which are enough to compute the area exactly.")

    # iris defaults to an Earth radius of 6367470 m where the cube has
    # no coordinate system, while cell_lengths uses the mean radius of
    # 6371000 m. That is a 0.11% difference in area, small but free to
    # avoid: pin the same sphere so lengths and areas agree.
    horizontal = horizontal.copy()
    guess_bounds(horizontal)
    if horizontal.coord("latitude").coord_system is None:
        sphere = iris.coord_systems.GeogCS(EARTH_RADIUS)
        horizontal.coord("latitude").coord_system = sphere
        horizontal.coord("longitude").coord_system = sphere
    return iris.analysis.cartography.area_weights(horizontal)


def bin_centres(bins):
    """Midpoints of a monotonic array of bin edges."""
    bins = np.asarray(bins, dtype=float)
    return 0.5 * (bins[1:] + bins[:-1])


@contextlib.contextmanager
def quiet_gsw():
    """Silence gsw's masked-array warning for the enclosed call.

    ``gsw`` emits ``'where' used without 'out', expect uninitialized
    memory in output`` every time it is handed a masked array, which is
    every time here.  The uninitialised values sit only underneath the
    mask and never reach the result -- asserted in
    ``validation/test_cosima_common.py`` -- but the warning fires once
    per call and buries everything else a notebook prints.

    Scoped to this one message so genuine gsw warnings still surface.
    """
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore", message=".*'where' used without 'out'.*",
            category=UserWarning)
        yield


def _require_gsw():
    if gsw is None:
        raise ImportError(
            "gsw (TEOS-10) is required for the density diagnostics; "
            "install it with `pixi add gsw`")


def pressure_from_depth(depth, lat):
    """Sea pressure in dbar from depth in metres (positive down)."""
    _require_gsw()
    depth = np.asarray(depth, dtype=float)
    lat = np.asarray(lat, dtype=float)
    return gsw.p_from_z(-depth, lat)


def absolute_salinity(practical_salinity, pressure, lon, lat):
    """Practical salinity (PSU) to absolute salinity (g/kg)."""
    _require_gsw()
    with quiet_gsw():
        return gsw.SA_from_SP(practical_salinity, pressure, lon, lat)


def conservative_temperature(abs_salinity, potential_temperature):
    """Potential temperature (degC) to conservative temperature (degC)."""
    _require_gsw()
    with quiet_gsw():
        return gsw.CT_from_pt(abs_salinity, potential_temperature)


def _broadcast_to_field(depth, lat, lon, shape):
    """Broadcast depth/lat/lon onto a ``(depth, lat, lon)`` field shape."""
    depth_b = np.broadcast_to(np.asarray(depth)[:, None, None], shape)
    lat_b = np.broadcast_to(np.asarray(lat)[None, :, None], shape)
    lon_b = np.broadcast_to(np.asarray(lon)[None, None, :], shape)
    return depth_b, lat_b, lon_b


def teos10_fields(thetao_cube, so_cube, temperature_is_conservative=False):
    """Absolute salinity, conservative temperature and pressure.

    Accepts fields with optional leading dimensions, such as time:
    ``(lat, lon)``, ``(depth, lat, lon)`` or
    ``(time, depth, lat, lon)``.  It returns masked arrays broadcast to
    the cube shape.  CMIP ``thetao`` is potential temperature, so it is
    converted unless the caller says otherwise (MOM5 ``temp`` is already
    conservative).
    """
    _require_gsw()
    salinity = masked_data(so_cube)
    temperature = masked_data(thetao_cube)

    if salinity.shape != temperature.shape:
        raise ValueError("thetao and so must have the same shape")

    def broadcast_1d(points, axis):
        """Broadcast a coordinate along its corresponding data axis."""
        reshape = [1] * salinity.ndim
        reshape[axis] = len(points)
        return np.broadcast_to(np.asarray(points).reshape(reshape),
                               salinity.shape)

    lat = lat_1d(so_cube)
    lon = lon_1d(so_cube)
    lat_axis = so_cube.coord_dims("latitude")[0]
    lon_axis = so_cube.coord_dims("longitude")[-1]
    depth_name = depth_coord_name(so_cube)
    if depth_name is None:
        depth_b = np.zeros(salinity.shape)
    else:
        depth = so_cube.coord(depth_name).points
        depth_axis = so_cube.coord_dims(depth_name)[0]
        depth_b = broadcast_1d(depth, depth_axis)
    lat_b = broadcast_1d(lat, lat_axis)
    lon_b = broadcast_1d(lon, lon_axis)
    pressure = pressure_from_depth(depth_b, lat_b)
    abs_sal = absolute_salinity(salinity, pressure, lon_b, lat_b)
    if temperature_is_conservative:
        cons_temp = temperature
    else:
        cons_temp = conservative_temperature(abs_sal, temperature)

    return abs_sal, cons_temp, pressure


DEFAULT_TEMPERATURE_BIN_WIDTH = 0.5


DEFAULT_SALINITY_BIN_WIDTH = 0.1


def ts_bin_edges(values, width):
    """Bin edges spanning the data, at the given width.

    Reproduces the original's construction: floor the minimum, ceil the
    maximum, pad by one bin either side.
    """
    finite = np.ma.compressed(np.ma.masked_invalid(values))
    if finite.size == 0:
        return np.array([0.0, width])
    low = np.floor(finite.min()) - width
    high = np.ceil(finite.max()) + width
    return np.arange(low, high + width, width)


def report_aliasing(histogram):
    """Warn when the bins are finer than the data can populate.

    Counts empty bins that sit *between* populated ones on the same
    temperature row.  Those gaps are not water that is absent, they are
    bins narrower than the spacing of the available values, and on the
    figure they appear as white streaks through the middle of the
    distribution.
    """
    filled = histogram > 0
    gaps = 0
    for row in filled:
        populated = np.where(row)[0]
        if populated.size > 1:
            gaps += (populated[-1] - populated[0] + 1) - populated.size
    if filled.sum() and gaps > 0.25 * filled.sum():
        LOGGER.warning(
            "%d of %d populated bins have an empty bin beside them within "
            "the occupied range; the bins are finer than the data can fill "
            "and the figure will show white streaks. Widen "
            "temperature_bin_width / salinity_bin_width, or keep the time "
            "axis so more values contribute.", gaps, int(filled.sum()))
    return gaps


def volume_weighted_ts_histogram(cons_temp, abs_sal, volume,
                                 temperature_bins, salinity_bins):
    """Volume-weighted joint T-S histogram.

    Returns an array of shape ``(len(temperature_bins) - 1,
    len(salinity_bins) - 1)`` holding the total volume in each cell.
    """
    cons_temp = np.ma.masked_invalid(np.ma.asarray(cons_temp, dtype=float))
    abs_sal = np.ma.masked_invalid(np.ma.asarray(abs_sal, dtype=float))
    volume = np.ma.masked_invalid(np.ma.asarray(volume, dtype=float))
    volume = np.ma.masked_array(
        np.broadcast_to(volume, cons_temp.shape),
        mask=np.broadcast_to(np.ma.getmaskarray(volume), cons_temp.shape))

    valid = (~np.ma.getmaskarray(cons_temp) & ~np.ma.getmaskarray(abs_sal)
             & ~np.ma.getmaskarray(volume))
    if not valid.any():
        return np.zeros((len(temperature_bins) - 1, len(salinity_bins) - 1))
    hist, _, _ = np.histogram2d(
        np.ma.filled(cons_temp, np.nan)[valid],
        np.ma.filled(abs_sal, np.nan)[valid],
        bins=[np.asarray(temperature_bins), np.asarray(salinity_bins)],
        weights=np.ma.filled(volume, 0.0)[valid])
    return hist

## Compute

In [ ]:
# The original's bin WIDTHS; the ranges come from the data.
TEMPERATURE_BIN_WIDTH = 0.5      # degC
SALINITY_BIN_WIDTH = 0.1         # g/kg

# Cell volume: volcello if the model has it, else thickness*area.
depth = thetao_cube.coord(depth_coord_name(thetao_cube))
if not depth.has_bounds():
    depth.guess_bounds()
thickness = np.diff(depth.bounds, axis=1)[:, 0]
volume = thickness[:, None, None] * cell_area(thetao_cube)[None]

# TEOS-10 for every month, not for a climatology.
pairs = [teos10_fields(t_slice, s_slice)[:2]
         for t_slice, s_slice
         in zip(thetao_cube.slices_over('time'),
                so_cube.slices_over('time'))]

TEMPERATURE_BINS = ts_bin_edges(
    np.ma.concatenate([ct.ravel() for _, ct in pairs]),
    TEMPERATURE_BIN_WIDTH)
SALINITY_BINS = ts_bin_edges(
    np.ma.concatenate([sa.ravel() for sa, _ in pairs]),
    SALINITY_BIN_WIDTH)

# Divide by the number of steps so the colour bar stays a
# volume rather than becoming volume x months.
histogram = sum(
    volume_weighted_ts_histogram(ct, sa, volume,
                                 TEMPERATURE_BINS, SALINITY_BINS)
    for sa, ct in pairs) / len(pairs)

gaps = report_aliasing(histogram)
print(f'{len(pairs)} months | bins '
      f'{len(TEMPERATURE_BINS)-1}x{len(SALINITY_BINS)-1} | '
      f'interior gaps {gaps}')
print(f'total volume: {histogram.sum():.3e} m3')

## Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
t_centres = bin_centres(TEMPERATURE_BINS)
s_centres = bin_centres(SALINITY_BINS)
plotted = np.ma.masked_less_equal(histogram, 0.0)

mesh = ax.pcolormesh(s_centres, t_centres, plotted, cmap='viridis',
                     shading='auto',
                     norm=mcolors.LogNorm(
                         vmin=max(np.ma.min(plotted), 1.0),
                         vmax=np.ma.max(plotted)))

# sigma2 contours over the T-S plane
s_grid, t_grid = np.meshgrid(s_centres, t_centres)
contours = ax.contour(s_grid, t_grid, gsw.sigma2(s_grid, t_grid),
                      colors='0.4', linewidths=0.6, levels=12)
ax.clabel(contours, inline=True, fontsize=7, fmt='%.1f')

ax.set_xlabel('Absolute salinity (g kg$^{-1}$)')
ax.set_ylabel('Conservative temperature ($^\circ$C)')
ax.set_title('Volume-weighted T-S diagram')
fig.colorbar(mesh, ax=ax, shrink=0.85, label='Volume (m$^3$)')
plt.show()

## Shut the cluster down

Workers hold their memory until the cluster closes. Leaving one running
in an idle kernel is the usual reason a later notebook cannot get the
memory it asks for.

In [ ]:
client.close()